# 03 — Integrating two datasets

**CIAD single-cell workshop**

Adapted from the Seurat *Introduction to scRNA-seq integration*:
<https://satijalab.org/seurat/articles/integration_introduction>

Everything so far used one sample. Real projects almost never do. The moment you
have two — two patients, two conditions, two runs on different days — you hit the
problem this notebook is about.

**The problem.** Cells cluster by *where they came from* instead of by *what they
are*. Two batches of B cells land in two separate clusters. Every downstream
answer is then wrong: you find the wrong number of cell types, and your
differential expression finds batch, not biology.

**The data.** Two PBMC datasets from 10x Genomics:

| | cells | chemistry |
|---|---|---|
| `pbmc3k` | ~2,700 | v1 (2016) |
| `pbmc_1k_v3` | ~1,200 | v3 (2018) |

Same tissue, same cell types, different technology and different day. That is a
real batch effect, not a simulated one — and unlike a two-condition dataset, we
know for certain that any separation by batch is technical, because the biology
is the same.

**What we do**

1. build one object from two datasets, and see the batch effect
2. correct it with **Harmony**, through Seurat's `IntegrateLayers()`
3. correct it with **Canek**, which works differently
4. compare the three results, by eye and with a number

About 50 minutes.

## Setup

This notebook uses a **different setup file** from notebooks 01 and 02.

It installs Canek in addition to Seurat. Canek depends on Bioconductor
packages, which take several minutes to install, so the earlier notebooks
deliberately leave them out rather than make everyone wait for something they
do not use.

Start this cell and read the introduction above while it runs.

In [ ]:
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup_canek.R")

In [ ]:
library(harmony)

cat("Canek  ", as.character(packageVersion("Canek")), "\n")
cat("harmony", as.character(packageVersion("harmony")), "\n")

## 1. Two datasets

Download both, read both, and keep track of which is which.

In [ ]:
# batch 1 — the pbmc3k data from notebooks 01 and 02
download.file("https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz",
              "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

# batch 2 — a later run, v3 chemistry
download.file("https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_filtered_feature_bc_matrix.tar.gz",
              "pbmc1k.tar.gz", quiet = TRUE)
untar("pbmc1k.tar.gz")

counts_v1 <- Read10X("filtered_gene_bc_matrices/hg19")
counts_v3 <- Read10X("filtered_feature_bc_matrix")

cat("v1:", nrow(counts_v1), "genes x", ncol(counts_v1), "cells\n")
cat("v3:", nrow(counts_v3), "genes x", ncol(counts_v3), "cells\n")

### The batch label

The single most important variable in this notebook. Everything downstream —
what we plot, what we correct, what we measure — is keyed on it.

In [ ]:
v1 <- CreateSeuratObject(counts_v1, project = "v1", min.cells = 3, min.features = 200)
v3 <- CreateSeuratObject(counts_v3, project = "v3", min.cells = 3, min.features = 200)

v1$batch <- "v1_3k"
v3$batch <- "v3_1k"

cat("v1:", ncol(v1), "cells\n")
cat("v3:", ncol(v3), "cells\n")

### Quality control, per batch

QC is done **within each batch**, never across. The two chemistries capture
different amounts of RNA, so a single count threshold would silently remove far
more cells from one batch than the other — which is itself a batch effect you
just created.

In [ ]:
v1[["percent.mt"]] <- PercentageFeatureSet(v1, pattern = "^MT-")
v3[["percent.mt"]] <- PercentageFeatureSet(v3, pattern = "^MT-")

VlnPlot(v1, features = c("nFeature_RNA", "percent.mt"), ncol = 2) +
  plot_annotation(title = "v1")


In [ ]:
VlnPlot(v3, features = c("nFeature_RNA", "percent.mt"), ncol = 2) +
  plot_annotation(title = "v3")

Look at the difference in `nFeature_RNA` before filtering. The v3 chemistry
detects far more genes per cell. That gap is the batch effect, visible before we
have done any analysis at all.

In [ ]:
before <- c(ncol(v1), ncol(v3))

v1 <- subset(v1, subset = nFeature_RNA > 200 & nFeature_RNA < 2500 & percent.mt < 5)
v3 <- subset(v3, subset = nFeature_RNA > 500 & nFeature_RNA < 5000 & percent.mt < 15)

cat("v1:", before[1], "->", ncol(v1), "cells\n")
cat("v3:", before[2], "->", ncol(v3), "cells\n")

### ✏️ Exercise 1

The thresholds above are different for the two batches — 200–2500 genes and 5%
mitochondrial for v1, 500–5000 and 15% for v3.

Justify or reject that choice. Look back at the two violin plots. Would a single
shared threshold have been more honest, or less?

Nothing to code. Write your answer in the next cell and be ready to defend it.

*Your answer:*



### One object, two layers

`merge()` combines the objects. In Seurat v5 the counts stay in **separate
layers**, one per batch — they are not silently pooled.

That layer structure is what `IntegrateLayers()` operates on later.

Note the gene intersection first. The two datasets were mapped to different
reference builds, so their gene lists differ. Merging without intersecting would
fill the missing genes with zeros and manufacture a difference between batches
that is purely bookkeeping.

In [ ]:
shared <- intersect(rownames(v1), rownames(v3))

cat("genes in v1    :", nrow(v1), "\n")
cat("genes in v3    :", nrow(v3), "\n")
cat("shared         :", length(shared), "\n")

pbmc <- merge(
  v1[shared, ],
  y = v3[shared, ],
  add.cell.ids = c("v1", "v3")
)

pbmc

In [ ]:
# two layers, one per batch
Layers(pbmc[["RNA"]])

table(pbmc$batch)

## 2. Seeing the batch effect

Run the standard pipeline, exactly as in notebook 02, and look at the result
coloured by batch.

Because the layers are separate, normalisation and variable feature selection are
done per layer and then combined — this is Seurat v5 handling the batch structure
for you.

In [ ]:
pbmc <- NormalizeData(pbmc, verbose = FALSE)
pbmc <- FindVariableFeatures(pbmc, verbose = FALSE)
pbmc <- ScaleData(pbmc, verbose = FALSE)
pbmc <- RunPCA(pbmc, verbose = FALSE)

pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "pca",
                reduction.name = "umap.unintegrated", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.unintegrated", group.by = "batch") +
  ggtitle("no integration")

There it is. The cells separate by batch, not by cell type.

To be sure that is technical rather than biological, check a marker. `MS4A1`
marks B cells, and both batches contain B cells — so if the B cells sit in two
separate places, the split is technical.

In [ ]:
FeaturePlot(pbmc, reduction = "umap.unintegrated",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4)

Each marker lights up in **two** places, once per batch. Same cell type,
split in two by technology alone. That is precisely what integration must fix.

### What clustering does with this

In [ ]:
pbmc <- FindNeighbors(pbmc, dims = 1:30, reduction = "pca", verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)

# how pure is each cluster, in batch terms?
round(prop.table(table(pbmc$seurat_clusters, pbmc$batch), margin = 1), 2)

Read that table as: for each cluster, what fraction came from each batch.

A cluster of 0.98 / 0.02 is a batch artefact — it is one batch's version of a
cell type. If you handed these clusters to a differential expression test you
would get a long list of significant genes that are really chemistry.

## 3. Harmony

`IntegrateLayers()` is Seurat v5's single entry point for integration. You pass it
a method, tell it which reduction to start from, and name the reduction to
create.

Harmony works **on the PCA embedding**. It does not touch expression values: it
takes the PCA coordinates and iteratively nudges cells so batches overlap while
cell type structure is preserved. The output is a new reduction, the same shape
as the PCA it started from.

In [ ]:
pbmc <- IntegrateLayers(
  object         = pbmc,
  method         = HarmonyIntegration,
  orig.reduction = "pca",
  new.reduction  = "harmony",
  verbose        = FALSE
)

Reductions(pbmc)

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "harmony",
                reduction.name = "umap.harmony", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.harmony", group.by = "batch") +
  ggtitle("Harmony")

## 4. Canek

A different approach, and a good contrast.

Canek identifies **mutual nearest neighbours** between batches — pairs of cells
that are each other's closest match across the batch boundary, which are taken to
be the same cell type. From those pairs it estimates the correction, using a
hybrid of a linear and a non-linear model.

Two practical differences from Harmony:

- Canek is not an `IntegrateLayers` method. It is called directly with
  `RunCanek()` and reads the batch from a **metadata column**, not from layers.
- We ask it to correct the **expression values**, producing a corrected assay,
  rather than the embedding. We then run PCA on that corrected assay ourselves.

That second point is the conceptual difference worth holding on to. Harmony hands
you corrected coordinates. Canek can hand you a corrected expression matrix,
which you can then analyse like any other assay.

Since Canek works from metadata rather than layers, we join the layers first.

In [ ]:
pbmc[["RNA"]] <- JoinLayers(pbmc[["RNA"]])

Layers(pbmc[["RNA"]])

In [ ]:
pbmc <- RunCanek(pbmc, batches = "batch",
                 correctEmbeddings = FALSE)

Assays(pbmc)

A new assay, `Canek`, holding corrected expression. The original `RNA` assay
is untouched — you can always go back.

Now the usual steps, but on the corrected assay, into a reduction of its own.

In [ ]:
DefaultAssay(pbmc) <- "Canek"

pbmc <- ScaleData(pbmc, verbose = FALSE)
pbmc <- RunPCA(pbmc, reduction.name = "pca.canek", verbose = FALSE)
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "pca.canek",
                reduction.name = "umap.canek", verbose = FALSE)

DefaultAssay(pbmc) <- "RNA"   # put it back, so later code is unsurprising

DimPlot(pbmc, reduction = "umap.canek", group.by = "batch") +
  ggtitle("Canek")

## 5. Comparing the three

Side by side, coloured by batch. What you want to see: the two colours mixed
everywhere, and the overall shape still showing distinct groups.

Both failures are visible in this plot. Too little correction leaves the batches
separate. Too much correction merges everything into one blob — batches perfectly
mixed, cell types destroyed.

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 5)

p_none <- DimPlot(pbmc, reduction = "umap.unintegrated", group.by = "batch") +
  ggtitle("none") + theme(legend.position = "none")
p_harm <- DimPlot(pbmc, reduction = "umap.harmony", group.by = "batch") +
  ggtitle("Harmony") + theme(legend.position = "none")
p_canek <- DimPlot(pbmc, reduction = "umap.canek", group.by = "batch") +
  ggtitle("Canek")

p_none + p_harm + p_canek

### Did the biology survive?

Mixing batches is easy — you could do it by shuffling the data. The test is
whether cell types are still distinct afterwards.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)

FeaturePlot(pbmc, reduction = "umap.harmony",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Harmony — each marker should now be in ONE place")

In [ ]:
FeaturePlot(pbmc, reduction = "umap.canek",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Canek")

### ✏️ Exercise 2

Pick one more marker pair and check it on both integrated UMAPs. Suggestions:
`GNLY` and `NKG7` for NK cells, `FCER1A` for dendritic cells, `CD8A` for CD8 T
cells.

Does the marker sit in one place, or two?

In [ ]:
# FeaturePlot(pbmc, reduction = "umap.harmony", features = c(______), ncol = 2)

## 6. Putting a number on it

Eyes are unreliable, and a UMAP is a projection. A simple, honest metric:

> For each cell, look at its 30 nearest neighbours in the corrected space. What
> fraction come from the *other* batch?

If batches are perfectly mixed, that fraction approaches the other batch's share
of the data. If batches stay separate, it approaches zero.

This is the idea behind published metrics like kBET and LISI, reduced to
something you can read in ten lines.

In [ ]:
mixing <- function(obj, reduction, batch = "batch", k = 30, dims = 1:30) {
  emb <- Embeddings(obj, reduction)[, dims]
  b   <- as.character(obj[[batch]][, 1])

  nn <- FNN::get.knn(emb, k = k)$nn.index
  # fraction of each cell's neighbours belonging to a different batch
  mean(rowMeans(matrix(b[nn], nrow = nrow(nn)) != b))
}

# what perfect mixing would look like: the chance two random cells differ
p  <- prop.table(table(pbmc$batch))
ideal <- 1 - sum(p^2)

cat(sprintf("%-14s %s\n", "method", "cross-batch neighbours"))
cat(sprintf("%-14s %.3f\n", "none",    mixing(pbmc, "pca")))
cat(sprintf("%-14s %.3f\n", "Harmony", mixing(pbmc, "harmony")))
cat(sprintf("%-14s %.3f\n", "Canek",   mixing(pbmc, "pca.canek")))
cat(sprintf("\n%-14s %.3f  (perfectly mixed)\n", "ideal", ideal))

### Read this carefully

Higher is more mixed. It is **not** simply better.

A method that destroyed all biological structure would score close to the ideal
and be useless. The number only tells you whether batches mixed; the marker plots
above tell you whether cell types survived. You need both, and neither alone is
evidence.

### ✏️ Exercise 3

Compute the same score **per cell type** rather than globally.

Cluster the Harmony embedding, then report the mixing score within each cluster.
A cluster that stays unmixed after integration is where the method failed — and
it is often a cell type genuinely present in only one batch.

Fill in the blank:

In [ ]:
pbmc <- FindNeighbors(pbmc, reduction = "harmony", dims = 1:30, verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)

round(prop.table(table(pbmc$seurat_clusters, pbmc$batch), margin = 1), 2)

# Which clusters are still dominated by one batch?
# threshold <- ______
# names(which(apply(prop.table(table(pbmc$seurat_clusters, pbmc$batch), 1), 1, max) > threshold))

### Clusters, before and after

The practical payoff. Compare this table to the one in section 2.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

DimPlot(pbmc, reduction = "umap.harmony",
        group.by = c("batch", "seurat_clusters"))

## 7. Which method should you use?

An honest answer: for a straightforward batch effect between runs of the same
tissue, most current methods work, and the differences you see here are smaller
than the differences caused by your QC thresholds.

What actually matters:

- **Integrate for visualisation and clustering. Do differential expression on the
  original data**, with the batch as a covariate. Corrected values have had
  variation removed by design, so p-values computed on them are not trustworthy.
  This is why we kept the `RNA` assay intact.
- **Check that biology survived.** Every time. A well-mixed UMAP proves nothing
  on its own.
- **Ask whether you should integrate at all.** If a cell type is genuinely
  present in one batch and absent from the other, integration will try to mix it
  anyway — and can invent correspondence that is not there.

### ✏️ Exercise 4

We integrated two batches of the *same* tissue, so we knew any separation was
technical.

Suppose instead the batches were *control* and *treated*, and the treatment
changed which cell types were present. What would integration do to that
difference, and how would you tell it apart from a batch effect?

Nothing to code. This is the question to think about before integrating your own
data.

*Your answer:*



## What we did

- Built one object from two real datasets and saw the batch effect in the raw
  data, in the UMAP, in the markers and in the cluster composition table.
- Corrected it two ways: Harmony on the embedding through `IntegrateLayers()`,
  Canek on the expression values through `RunCanek()`.
- Compared them by eye and with a neighbour-mixing score, and said explicitly why
  that score is not sufficient on its own.

---

### Answers

<details>
<summary>Click to expand</summary>

**Exercise 1**

Different thresholds are the right call, and the violin plots are the
justification: v3 chemistry detects roughly twice as many genes per cell. A
shared upper bound of 2,500 genes would remove a large fraction of healthy v3
cells and almost no v1 cells — turning a QC step into a batch effect. QC asks
"is this a real cell", and what counts as real depends on the chemistry.

The honest version of "one shared threshold" is a quantile: keep the middle 95%
*within each batch*.

**Exercise 2**

```r
FeaturePlot(pbmc, reduction = "umap.harmony", features = c("GNLY", "NKG7"), ncol = 2)
```

**Exercise 3**

```r
threshold <- 0.9
comp <- prop.table(table(pbmc$seurat_clusters, pbmc$batch), 1)
names(which(apply(comp, 1, max) > threshold))
```

Any cluster still above 0.9 after integration is worth looking at directly.
Sometimes the method failed; sometimes the cell type really is in one batch only,
in which case leaving it unmixed is the correct behaviour.

**Exercise 4**

Integration cannot distinguish "this batch differs technically" from "this batch
differs biologically" — it only sees that the batches differ, and its job is to
remove that. If treatment changed the cell type composition, integration will
push those cells together and can hide the effect you are studying.

Ways to tell them apart:

- Cell types shared by both conditions should align; a genuinely
  condition-specific population should not. If *everything* aligns perfectly,
  suspect over-correction.
- Keep the uncorrected data and check whether the difference is visible there.
- Do the statistics on uncorrected counts with condition as a covariate.
  Integration is for seeing, not for testing.

</details>